# Task 3 — Usage E8 on teacher + rare images

Train **one nine-class Usage model** from scratch on the combined dataset. Run All trains all five saved folds, one at a time.

The recipe comes from **E8 SmallCNN + weighted cross-entropy + small image shifts**. Its saved teacher-only pooled macro-F1 was **0.4194**, the best completed five-fold single GPU model found in the saved Usage comparison. We reuse the recipe and start with fresh weights.

| Control | Fixed value |
|---|---|
| Input | RGB, 60 pixels wide × 80 pixels high |
| CNN widths | 32, 64, 128, 256; average pooling |
| Epochs / batch | 30 / 128 |
| Optimizer | AdamW; learning rate 0.001; weight decay 0.0001 |
| Schedule | Cosine; final learning rate 0.00001 |
| Loss | Effective-number weighted cross-entropy; beta 0.999, cap 5 |
| Image shift | Up to 2 pixels, with probability 0.5 |
| Seed / precision | 2753 / FP32 |
| Checkpoint | Final epoch; no early stopping |

**Before Run All:** choose a GPU in **Runtime → Change runtime type**. Put `teacher_plus_rare_usage_training.zip` in `MyDrive/MLA2/data/`. Keep the existing `task3-data.zip` in that same folder.

The training ZIP contains a frozen copy of the code, combined split, added images and E8 evidence. It runs in its own Colab folder, so uncommitted local code does not need to be pushed first.


## 1. Mount Drive and unpack the training bundle

The bundle is checked file by file before training. Keep this notebook in the same Colab session while the five folds run. Completed folds can be reused after a disconnect; an unfinished fold restarts from scratch.


In [1]:
from pathlib import Path
import hashlib
import json
import os
import shutil
import sys
import zipfile

from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
drive.mount(str(DRIVE_MOUNT), force_remount=False)
DRIVE_PROJECT = DRIVE_MOUNT / "MyDrive/MLA2"
BUNDLE_ZIP = DRIVE_PROJECT / "data/teacher_plus_rare_usage_training.zip"
TEACHER_ZIP = DRIVE_PROJECT / "data/task3-data.zip"
DRIVE_TASK_DIR = DRIVE_PROJECT / "task3_usage_expanded_e8"
DRIVE_REGISTRY = DRIVE_TASK_DIR / "experiments/t3_usage_expanded_e8/usage/results/runs.csv"

def sha256_file(path):
    with Path(path).open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

def safe_members(archive, prefix=None):
    members = archive.infolist()
    for member in members:
        name = member.filename
        parts = Path(name).parts
        if Path(name).is_absolute() or ".." in parts or "\\" in name:
            raise ValueError(f"Unsafe archive path: {name}")
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError(f"Archive symlink is not allowed: {name}")
    return [m for m in members if prefix is None or m.filename.startswith(prefix)]

if not BUNDLE_ZIP.is_file():
    raise FileNotFoundError(f"Upload the training ZIP here first: {BUNDLE_ZIP}")
bundle_digest = sha256_file(BUNDLE_ZIP)
LOCAL_BUNDLE = Path("/content") / f"usage-training-{bundle_digest[:12]}.zip"
if not LOCAL_BUNDLE.is_file() or sha256_file(LOCAL_BUNDLE) != bundle_digest:
    shutil.copyfile(BUNDLE_ZIP, LOCAL_BUNDLE)
if sha256_file(LOCAL_BUNDLE) != bundle_digest:
    raise RuntimeError("Training ZIP copy is incomplete. Run this cell again.")
REPO_DIR = Path("/content") / f"MLA2-usage-expanded-{bundle_digest[:12]}"
with zipfile.ZipFile(LOCAL_BUNDLE) as archive:
    members = safe_members(archive)
    manifest = json.loads(archive.read("training_bundle_manifest.json"))
    expected = manifest["files"]
    actual = {m.filename for m in members if not m.is_dir()}
    if actual != set(expected) | {"training_bundle_manifest.json"}:
        raise ValueError("The training archive inventory differs from its manifest.")
    for name, digest in expected.items():
        if hashlib.sha256(archive.read(name)).hexdigest() != digest:
            raise ValueError(f"Training archive file changed: {name}")
    if not REPO_DIR.exists():
        REPO_DIR.mkdir()
        archive.extractall(REPO_DIR)
for name, digest in expected.items():
    if not (REPO_DIR / name).is_file() or sha256_file(REPO_DIR / name) != digest:
        raise RuntimeError(f"Local bundle file differs: {name}. Use a fresh runtime.")

# The verified archive stays intact. Current code comes from GitHub.
BUNDLE_ROOT = REPO_DIR
import subprocess
import tempfile

REPOSITORY = "https://github.com/TrnLin/MLA2.git"
BRANCH = "fashion-analysis-and-cleanup"
with tempfile.TemporaryDirectory(prefix="usage-current-code-", dir="/content") as temporary:
    checkout = Path(temporary) / "project"
    subprocess.run(["git", "clone", "--quiet", "--depth=1", "--branch", BRANCH,
                    REPOSITORY, str(checkout)], check=True)
    CODE_COMMIT = subprocess.check_output(
        ["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
    REPO_DIR = Path(str(BUNDLE_ROOT) + "-code-" + CODE_COMMIT[:12])
    if REPO_DIR.exists():
        subprocess.run(["git", "-C", str(REPO_DIR), "diff", "--quiet", "HEAD", "--", "src"],
                       check=True)
    else:
        checkout.rename(REPO_DIR)
print("Current code commit:", CODE_COMMIT)

if "fashion.config" in sys.modules:
    loaded_root = Path(sys.modules["fashion.config"].ROOT)
    if loaded_root != REPO_DIR:
        raise RuntimeError("Another training notebook is loaded. Restart the session, then Run All.")
os.chdir(REPO_DIR)
os.environ["FASHION_PROJECT_ROOT"] = str(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
from fashion.train.task3_bundle_inputs import install_bundle_inputs
print("Reused data/reference files:", install_bundle_inputs(BUNDLE_ROOT, REPO_DIR))
LOCAL_REGISTRY = REPO_DIR / "results/runs.csv"
print("Frozen code and data ready:", REPO_DIR)
print("Bundle SHA-256:", bundle_digest)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Frozen code and data ready: /content/MLA2-usage-expanded-464bbdb20def
Bundle SHA-256: 464bbdb20defb8e205a1910a607313fb5c2eccf8b7c998648c5722c4c768a856


## 2. Load teacher images onto Colab disk

The original teacher split stays fixed. All 120 added images use the same Usage target and class map. Normalization and class weights are fitted on each combined training fold only.

The manifest contains 38,732 rows. Training uses 32,892 development rows with valid Usage labels. Holdout, quarantine and prediction images are not used for fitting or this comparison.


In [2]:
if not TEACHER_ZIP.is_file():
    raise FileNotFoundError(f"Existing teacher archive is missing: {TEACHER_ZIP}")
LOCAL_TEACHER_ZIP = Path("/content/task3-data.zip")
if (not LOCAL_TEACHER_ZIP.is_file()
        or LOCAL_TEACHER_ZIP.stat().st_size != TEACHER_ZIP.stat().st_size):
    partial = LOCAL_TEACHER_ZIP.with_suffix(".zip.partial")
    shutil.copyfile(TEACHER_ZIP, partial)
    if partial.stat().st_size != TEACHER_ZIP.stat().st_size:
        raise RuntimeError("Teacher ZIP copy is incomplete. Run this cell again.")
    partial.replace(LOCAL_TEACHER_ZIP)
with zipfile.ZipFile(LOCAL_TEACHER_ZIP) as archive:
    members = safe_members(archive, prefix="data/raw/teacher/")
    if not members:
        raise ValueError("Teacher archive must contain data/raw/teacher/.")
    for member in members:
        destination = REPO_DIR / member.filename
        if member.is_dir():
            destination.mkdir(parents=True, exist_ok=True)
        elif not destination.is_file() or destination.stat().st_size != member.file_size:
            archive.extract(member, REPO_DIR)
print("Teacher images are on local disk. The next step checks training image hashes.")


Teacher images are on local disk. The next step checks training image hashes.


## 3. Check the data, recipe and old GPU evidence

This checks every training image, the saved split hashes, teacher fold preservation, class weights and E8's saved probabilities. It never loads E8 weights into the new model.

No package upgrade is needed in the usual Colab GPU runtime. The run records the actual Python, PyTorch and library versions.


In [3]:
import pandas as pd
import torch
from IPython.display import display
from fashion.train.task3_usage_expanded import (
    E8_DIRECTORY, check_e8_sources, expanded_usage_spec,
    run_expanded_usage, validate_dataset,
)

if not torch.cuda.is_available():
    raise RuntimeError("Choose a GPU runtime, then Run All.")
splits, contract = validate_dataset(root=REPO_DIR)
E8_DIR = REPO_DIR / "results/evidence/task3" / E8_DIRECTORY
SOURCE_REGISTRY = REPO_DIR / "reference/e8_runs.csv"
sources = check_e8_sources(
    directory=E8_DIR, registry_path=SOURCE_REGISTRY, root=REPO_DIR,
)
print("Usage output folder:", DRIVE_TASK_DIR)
print("Usage-only run log:", DRIVE_REGISTRY)
print("GPU:", torch.cuda.get_device_name(0))
print("Training images and five E8 folds verified.")
display(pd.DataFrame(contract["folds"]))
display(pd.Series(expanded_usage_spec().to_dict(), name="Frozen recipe"))


Usage output folder: /content/drive/MyDrive/MLA2/task3_usage_expanded_e8
Usage-only run log: /content/drive/MyDrive/MLA2/task3_usage_expanded_e8/experiments/t3_usage_expanded_e8/usage/results/runs.csv
GPU: NVIDIA A100-SXM4-40GB
Training images and five E8 folds verified.


,fold,training_rows,validation_rows,added_training_rows,added_validation_rows
0,0,26315,6577,96,24
1,1,26313,6579,97,23
2,2,26314,6578,95,25
3,3,26316,6576,97,23
4,4,26310,6582,95,25


,Frozen recipe
name,usage_expanded_e8
target,usage
experiment_id,t3_usage_expanded_e8
hypothesis_id,t3_usage_teacher_plus_rare_images
artifact_dir,experiments/t3_usage_expanded_e8
run_prefix,t3_usage_expanded_e8
changed_factor,training_dataset
training_augmentation,translation_uniform_2px_p05
loss_name,effective_number_cross_entropy
parent_artifact_dir,experiments/t3_usage_e8_translation


## 4. Train the five folds

Each fold starts with fresh weights and runs 30 epochs. Its registry row is written before the first optimizer step. Checkpoints, learning curves, clean training predictions, validation predictions and corruption checks are saved to Drive.

Retraining saves all model files and the run log in the separate Drive folder `MyDrive/MLA2/task3_usage_expanded_e8/`. The log is under `experiments/t3_usage_expanded_e8/usage/results/runs.csv` inside that folder. This starts a new set of runs; it does not import earlier runs from the old `task3` folder.

Run All defaults to all five folds. Keep `FOLDS` unchanged for the full E8 comparison. After a disconnect, completed runs in the new folder are reused only if their recipe, dataset and saved files still match.


In [4]:
FOLDS = (0, 1, 2, 3, 4)
result = run_expanded_usage(
    root=REPO_DIR,
    output_root=DRIVE_TASK_DIR,
    registry_path=DRIVE_REGISTRY,
    registry_mirrors=(LOCAL_REGISTRY,),
    e8_directory=E8_DIR,
    source_registry_path=SOURCE_REGISTRY,
    folds=FOLDS,
    resume=True,
)
print("Saved teacher-only comparison:", result["comparison_path"])


[task3] preparing target=usage fold=0: train=26,315 (before selection=26,315), validation=6,577
[task3] fitting fold-training RGB statistics for target=usage fold=0
[task3] RGB statistics ready for target=usage fold=0
[task3] registered t3_usage_expanded_e8_usage_smallcnn_f0_s2753_6eb56854d557_20260906T091936Zd9a955; the first optimiser step may now run
[task3] target=usage fold=0 epoch=1/30 train_loss=1.2735 train_macro_f1=0.1559 validation_loss=1.3602 validation_macro_f1=0.1399
[task3] target=usage fold=0 epoch=2/30 train_loss=1.0178 train_macro_f1=0.2657 validation_loss=1.2422 validation_macro_f1=0.2117
[task3] target=usage fold=0 epoch=3/30 train_loss=0.9200 train_macro_f1=0.2884 validation_loss=0.9764 validation_macro_f1=0.2772
[task3] target=usage fold=0 epoch=4/30 train_loss=0.8218 train_macro_f1=0.3413 validation_loss=0.8520 validation_macro_f1=0.3802
[task3] target=usage fold=0 epoch=5/30 train_loss=0.7979 train_macro_f1=0.3622 validation_loss=0.8869 validation_macro_f1=0.2797

## 5. Read the fair comparison

Compare the new model and E8 on **the exact same teacher validation images**. The combined score includes added-source images with a different class mix, so it cannot replace that comparison.

The added set has 97 Party, 16 Home, 6 Smart Casual and 1 Travel image. It adds no NA images. Smart Casual has only two added product families; Travel has one image. A high score on those few images is weak evidence.

Review the teacher class scores, clean training gap and corruption checks before choosing a final model. This notebook does not change the chosen model or open the holdout.


In [5]:
summary = result["comparison"]
teacher = summary["sources"]["teacher"]["metrics"]
old = summary["teacher_reference_e8"]
print(f"Teacher-only macro-F1: E8 {old['macro_f1']:.4f} → expanded {teacher['macro_f1']:.4f}")
print(f"Change: {summary['teacher_macro_f1_change']:+.4f}")
display(pd.DataFrame(summary["teacher_per_class"]))
display(pd.DataFrame([
    {"scope": name, "rows": scope["rows"],
     "macro_f1": scope["metrics"]["macro_f1"] if scope["metrics"] else None}
    for name, scope in summary["sources"].items()
]))
display(pd.DataFrame([
    {"run_id": run["run_id"],
     "teacher_training_f1": run["metrics"]["source_metrics"]["clean_training"]["teacher"]["metrics"]["macro_f1"],
     "teacher_validation_f1": run["metrics"]["source_metrics"]["validation"]["teacher"]["metrics"]["macro_f1"]}
    for run in result["fold_results"]
]))
print("Results:", Path(result["comparison_path"]).parent)
print("Run registry:", DRIVE_REGISTRY)


Teacher-only macro-F1: E8 0.4194 → expanded 0.4086
Change: -0.0108


,class_name,support,e8_f1,expanded_f1,change
0,Casual,25151,0.930523,0.929725,-0.000797
1,Ethnic,2183,0.847377,0.850199,0.002821
2,Formal,1949,0.777070,0.769765,-0.007305
3,Home,1,0.000000,0.000000,0.000000
4,NA,61,0.169014,0.222222,0.053208
5,Party,12,0.076923,0.000000,-0.076923
6,Smart Casual,47,0.103448,0.038462,-0.064987
7,Sports,3346,0.666102,0.663253,-0.002849
8,Travel,22,0.204082,0.204082,0.000000


,scope,rows,macro_f1
0,combined,32892,0.554769
1,teacher,32772,0.408634
2,added,120,0.170765


,run_id,teacher_training_f1,teacher_validation_f1
0,t3_usage_expanded_e8_usage_smallcnn_f0_s2753_6...,0.783422,0.406567
1,t3_usage_expanded_e8_usage_smallcnn_f1_s2753_6...,0.805734,0.384892
2,t3_usage_expanded_e8_usage_smallcnn_f2_s2753_6...,0.795581,0.429376
3,t3_usage_expanded_e8_usage_smallcnn_f3_s2753_6...,0.810845,0.406298
4,t3_usage_expanded_e8_usage_smallcnn_f4_s2753_6...,0.687481,0.402142


Results: /content/drive/MyDrive/MLA2/task3_usage_expanded_e8/experiments/t3_usage_expanded_e8/usage/aggregate
Run registry: /content/drive/MyDrive/MLA2/task3_usage_expanded_e8/experiments/t3_usage_expanded_e8/usage/results/runs.csv
